# Road Safety Dissertation Coding Notebook

**Central research question:**  
*To what extent can machine learning predict serious or fatal outcomes in reported road traffic collisions in Great Britain?*

This notebook mirrors the standalone Python script. It is designed for dissertation documentation and reproducibility.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("road_safety_analysis/analysis_ready_road_safety.csv")
OUTPUT_DIR = Path("road_safety_coding_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "tables").mkdir(exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)
TARGET = "serious_or_fatal"


## 1. Load analysis-ready dataset

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df.shape


## 2. Descriptive statistics

In [ ]:
summary = {
    "records": len(df),
    "year_min": int(df["collision_year"].min()),
    "year_max": int(df["collision_year"].max()),
    "serious_or_fatal_count": int(df[TARGET].sum()),
    "slight_count": int((df[TARGET] == 0).sum()),
    "serious_or_fatal_rate_pct": round(df[TARGET].mean() * 100, 2)
}
summary


In [ ]:
severity = df.groupby("collision_severity").agg(collisions=("collision_index", "count")).reset_index()
severity["percentage"] = (severity["collisions"] / severity["collisions"].sum() * 100).round(2)
severity


In [ ]:
yearly = df.groupby("collision_year").agg(
    collisions=("collision_index", "count"),
    serious_or_fatal=(TARGET, "sum")
).reset_index()
yearly["serious_fatal_rate_pct"] = (yearly["serious_or_fatal"] / yearly["collisions"] * 100).round(2)
yearly


## 3. Train-test split and feature selection

In [ ]:
numeric_features = [
    "month", "hour", "is_weekend", "is_night", "longitude", "latitude",
    "number_of_vehicles", "speed_limit", "traffic_link_length_km",
    "traffic_all_motor_vehicles", "traffic_all_motor_vehicles_per_km",
    "traffic_cars_taxis_share", "vehicle_record_count", "n_pedal_cycles",
    "n_motorcycles", "n_cars_taxis", "n_buses_minibuses", "n_goods_vehicles",
    "vehicle_type_nunique", "mean_driver_age", "min_driver_age", "max_driver_age",
    "any_young_driver_17_24", "any_older_driver_65_plus", "mean_vehicle_age",
    "max_vehicle_age"
]
categorical_features = [
    "day_of_week", "police_force", "local_authority_highway", "urban_or_rural_area",
    "first_road_class", "road_type", "junction_detail", "junction_control",
    "pedestrian_crossing", "light_conditions", "weather_conditions",
    "road_surface_conditions", "special_conditions_at_site", "carriageway_hazards",
    "trunk_road_flag"
]
numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]
features = numeric_features + categorical_features

train_df = df[df["collision_year"].between(2020, 2023)].copy()
test_df = df[df["collision_year"] == 2024].copy()

# Dissertation feasibility: stratified training sample.
sample_size = 15000
train_sample = train_df.groupby(TARGET, group_keys=False).apply(
    lambda x: x.sample(n=max(1, int(sample_size * len(x) / len(train_df))), random_state=42)
).sample(frac=1, random_state=42)

X_train, y_train = train_sample[features].copy(), train_sample[TARGET].astype(int)
X_test, y_test = test_df[features].copy(), test_df[TARGET].astype(int)
len(X_train), len(X_test)


## 4. Preprocessing and models

In [ ]:
for col in categorical_features:
    X_train[col] = X_train[col].astype("object").where(~X_train[col].isna(), "missing").astype(str)
    X_test[col] = X_test[col].astype("object").where(~X_test[col].isna(), "missing").astype(str)

numeric_pipe_scaled = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
numeric_pipe_unscaled = Pipeline([("imputer", SimpleImputer(strategy="median"))])
ohe = OneHotEncoder(handle_unknown="ignore")
categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", ohe)])

preprocess_scaled = ColumnTransformer([
    ("numeric", numeric_pipe_scaled, numeric_features),
    ("categorical", categorical_pipe, categorical_features)
])
preprocess_unscaled = ColumnTransformer([
    ("numeric", numeric_pipe_unscaled, numeric_features),
    ("categorical", categorical_pipe, categorical_features)
])

models = {
    "Dummy majority baseline": Pipeline([
        ("preprocess", preprocess_unscaled),
        ("model", DummyClassifier(strategy="most_frequent"))
    ]),
    "Logistic Regression (SGD)": Pipeline([
        ("preprocess", preprocess_scaled),
        ("model", SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42, max_iter=1000))
    ]),
    "Random Forest": Pipeline([
        ("preprocess", preprocess_unscaled),
        ("model", RandomForestClassifier(n_estimators=100, max_depth=16, min_samples_leaf=50,
                                         class_weight="balanced_subsample", n_jobs=-1, random_state=42))
    ])
}


## 5. Model evaluation

In [ ]:
def evaluate(name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": name,
        "threshold": threshold,
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
        "roc_auc": round(roc_auc_score(y_true, y_prob), 4) if len(np.unique(y_prob)) > 1 else np.nan,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)
    }

results = []
probs = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    probs[name] = y_prob
    results.append(evaluate(name, y_test, y_prob, threshold=0.5))

performance = pd.DataFrame(results)
performance


## 6. Robustness: threshold sensitivity

In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
threshold_table = pd.DataFrame([
    evaluate("Random Forest", y_test, probs["Random Forest"], threshold=t)
    for t in thresholds
])
threshold_table
